# **EXTRACT**

In [ ]:
import pandas as pd

df = pd.read_csv('BootCamp_Ciencia_de_Dados.csv')
user_ids = df['UserID'].tolist()
print(user_ids)

In [ ]:
bootcamp_api_url = "http://127.0.0.1:8000"

In [ ]:
!pip install fastapi uvicorn -q
from google.colab import files
files.upload()

In [ ]:
%%writefile api.py
import json
import sqlite3

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="API Santander (local)")
DB = "banco.db"


def conectar():
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    return con


with conectar() as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            card_number TEXT NOT NULL UNIQUE,
            balance REAL NOT NULL DEFAULT 0,
            news TEXT NOT NULL DEFAULT '[]'
        )
    """)


class NovoUsuario(BaseModel):
    name: str
    card_number: str


class Noticia(BaseModel):
    description: str


def para_dict(row):
    return {
        "id": row["id"],
        "name": row["name"],
        "account": {"balance": row["balance"]},
        "card": {"number": row["card_number"]},
        "news": json.loads(row["news"]),
    }


@app.post("/users", status_code=201)
def criar_usuario(dados: NovoUsuario):
    try:
        with conectar() as con:
            cur = con.execute(
                "INSERT INTO users (name, card_number) VALUES (?, ?)",
                (dados.name, dados.card_number),
            )
            row = con.execute("SELECT * FROM users WHERE id = ?", (cur.lastrowid,)).fetchone()
    except sqlite3.IntegrityError:
        raise HTTPException(status_code=409, detail="Cartão já cadastrado.")
    return para_dict(row)


@app.get("/users")
def listar_usuarios():
    with conectar() as con:
        return [para_dict(r) for r in con.execute("SELECT * FROM users")]


@app.get("/users/{user_id}")
def buscar_usuario(user_id: int):
    with conectar() as con:
        row = con.execute("SELECT * FROM users WHERE id = ?", (user_id,)).fetchone()
    if row is None:
        raise HTTPException(status_code=404, detail="Usuário não encontrado.")
    return para_dict(row)


@app.post("/users/{user_id}/news")
def adicionar_noticia(user_id: int, noticia: Noticia):
    with conectar() as con:
        row = con.execute("SELECT * FROM users WHERE id = ?", (user_id,)).fetchone()
        if row is None:
            raise HTTPException(status_code=404, detail="Usuário não encontrado.")
        lista = json.loads(row["news"]) + [noticia.model_dump()]
        con.execute("UPDATE users SET news = ? WHERE id = ?", (json.dumps(lista), user_id))
    return {"id": user_id, "news": lista}

In [ ]:
import subprocess, time, os

print("Arquivos na pasta:", os.listdir("."))

log = open("servidor.log", "w")
servidor = subprocess.Popen(
    ["python", "-m", "uvicorn", "api:app", "--port", "8000"],
    stdout=log, stderr=log,
)
time.sleep(5)

if servidor.poll() is None:
    print("Servidor ligado e rodando")
else:
    print("O servidor caiu. Motivo:")
    print(open("servidor.log").read())

In [ ]:
import requests

bootcamp_api_url = "http://127.0.0.1:8000"

novos_usuarios = [
    {"name": "Ana Pereira", "card_number": "xxxx xxxx xxxx 4444"},
    {"name": "Bruno Alves", "card_number": "xxxx xxxx xxxx 5555"},
    {"name": "Carla Mendes", "card_number": "xxxx xxxx xxxx 6666"},
    {"name": "Diego Rocha", "card_number": "xxxx xxxx xxxx 7777"},
    {"name": "Elisa Martins", "card_number": "xxxx xxxx xxxx 8888"},
    {"name": "João Nogueira", "card_number": "xxxx xxxx xxxx 9999"},
    {"name": "Evelyn Balbino", "card_number": "4444"},
    {"name": "Carla Mendes", "card_number": "6666"},
    {"name": "Diego Rocha", "card_number": "7777"},
    {"name": "Elisa Martins", "card_number": "8888"},
    {"name": "João Nogueira", "card_number": "9999"},
]
for usuario in novos_usuarios:
    r = requests.post(f"{bootcamp_api_url}/users", json=usuario)
    print(r.status_code, r.json())

In [ ]:
import requests
import json

def get_user(id):
  response = requests.get(f'{bootcamp_api_url}/users/{id}')
  return response.json() if response.status_code == 200 else None

users = [user for id in user_ids if (user := get_user(id)) is not None]
print(json.dumps(users, indent=2))

# **TRANSFORM**

In [ ]:
!pip install openai

In [ ]:
from google.colab import userdata

open_api_key = userdata.get("OPENAI_API_KEY").strip()

print(len(open_api_key), open_api_key.startswith("sk-"))

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=open_api_key)


def generate_ai_news(user):
    completion = client.chat.completions.create(
        model="gpt-6-luna",
        messages=[
            {
                "role": "system",
                "content": "Você é um especialista em marketing bancário."
            },

            {"role": "user",
             "content": f"Crie uma mensagem para {user['name']} sobre a importância dos investimentos (máximo de 100 caracteres)."
            },
        ],
    )
    return completion.choices[0].message.content.strip('\"')

for user in users:
    news = generate_ai_news(user)
    print(news)
    user['news'].append({
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": news,
    })

# **LOAD**

*Rodar o codigo abaixo uma unica vez*

In [ ]:
%%writefile -a api.py


class NoticiaPut(BaseModel):
    icon: str | None = None
    description: str


class UsuarioPut(BaseModel):
    news: list[NoticiaPut]


@app.put("/users/{user_id}")
def atualizar_usuario(user_id: int, dados: UsuarioPut):
    with conectar() as con:
        row = con.execute("SELECT * FROM users WHERE id = ?", (user_id,)).fetchone()
        if row is None:
            raise HTTPException(status_code=404, detail="Usuário não encontrado.")
        news = [n.model_dump() for n in dados.news]
        con.execute("UPDATE users SET news = ? WHERE id = ?", (json.dumps(news), user_id))
        row = con.execute("SELECT * FROM users WHERE id = ?", (user_id,)).fetchone()
    return para_dict(row)

In [ ]:
import subprocess, time

# desliga qualquer servidor antigo na porta 8000
subprocess.run(["pkill", "-f", "uvicorn"])
time.sleep(2)

log = open("servidor.log", "w")
servidor = subprocess.Popen(
    ["python", "-m", "uvicorn", "api:app", "--port", "8000"],
    stdout=log, stderr=log,
)
time.sleep(5)

if servidor.poll() is None:
    print("Servidor ligado e rodando")
else:
    print("O servidor caiu. Motivo:")
    print(open("servidor.log").read())

In [ ]:
def update_user(user):
  response = requests.put(f"{bootcamp_api_url}/users/{user['id']}", json=user)
  return True if response.status_code == 200 else False

for user in users:
  success = update_user(user)
  print(f"User {user['name']} updated? {success}!")